# 🇹🇭 OpenThai-NER: Final Training & Production Pipeline

Notebook สำหรับเทรนโมเดล **OpenThai-NER** (Fine-tuning บน `Pavarissy/phayathaibert-thainer`)
- แก้ปัญหา `eval_loss: NaN` อย่างถาวร (bf16/fp32 + gradient clipping + label masking -100)
- คลีน Dataset และ Normalize Label Schema (แก้ `DTAE` -> `DATE`, `ORG` -> `ORGANIZATION`)
- วัดผลด้วย `seqeval` (Entity-level F1 / Precision / Recall)
- Export เป็น ONNX INT8 สำหรับ Production

In [ ]:
# 1. ติดตั้ง Dependencies
!pip install -q transformers datasets seqeval evaluate sentencepiece accelerate onnx onnxruntime gradio

In [ ]:
# 2. ตรวจสอบสถานะ GPU
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('bf16 Supported:', torch.cuda.is_bf16_supported())

In [ ]:
# 3. รัน Dataset Cleaning & Stratified Splitting
!python scripts/clean_dataset.py

In [ ]:
# 4. เริ่มเทรนโมเดล OpenThai-NER (3 Epochs)
!python train_ner.py --epochs 3 --batch_size 16 --learning_rate 2e-5 --output_dir models/openthai-ner-final

In [ ]:
# 5. ประเมินผล Entity-Level F1 บน Test Set
!python scripts/evaluate_benchmark.py --model models/openthai-ner-final --test_file data/test.jsonl

In [ ]:
# 6. Export เป็น ONNX และทำ INT8 Quantization
!python scripts/export_onnx.py --model models/openthai-ner-final --output_dir models/onnx

In [ ]:
# 7. Push โมเดลขึ้น Hugging Face Hub (เมื่อเทรนเสร็จสมบูรณ์)
from huggingface_hub import login
# login()  # ใส่ Hugging Face Write Token
# from transformers import AutoModelForTokenClassification, AutoTokenizer
# model = AutoModelForTokenClassification.from_pretrained('models/openthai-ner-final')
# tokenizer = AutoTokenizer.from_pretrained('models/openthai-ner-final')
# model.push_to_hub('JonusNattapong/OpenThai-NER')
# tokenizer.push_to_hub('JonusNattapong/OpenThai-NER')